# Sesión: Orquestación Cíclica (LangGraph) y Memoria a Largo Plazo

## Objetivo del taller
En la sesión anterior construimos un Copiloto de Operaciones con un flujo lineal (**Planner $\rightarrow$ HITL $\rightarrow$ Executor**). Hoy vamos a llevarlo a nivel de producción resolviendo tres retos reales:

1. **Orquestación Cíclica con Autocorrección (`StateGraph`):** Si el agente generador (`Drafter`) incumple una regla de `SKILL.md`, supera el stock o ignora una preferencia corporativa, un nodo `Auditor` rechazará el borrador y devolverá el control al `Drafter` con instrucciones precisas de corrección (hasta un máximo de 3 ciclos).
2. **Estado Persistente y HITL Real (`SqliteSaver` + `interrupt()`):** Guardaremos cada transición del grafo en SQLite (`langgraph_checkpoints.db`). Podrás pausar la ejecución ante una aprobación humana, **reiniciar el Kernel de Python** y reanudar exactamente donde se quedó.
3. **Context Engineering y Memoria en 3 Niveles:**
   - **Corto plazo:** Estado activo del turno actual.
   - **Medio plazo (*Context Engineering*):** Compresión automática mediante *Rolling Summary* + poda de mensajes antiguos (`RemoveMessage`) para evitar que la ventana de contexto explote.
   - **Largo plazo (`long_term_memory.db`):** Almacén persistente entre sesiones con **memoria semántica** (reglas y preferencias permanentes) y **memoria episódica** (decisiones pasadas).

---

In [ ]:
# Configuración de Gemini, Base de Datos Corporativa y Skill (Ejecutar)
import os
import json
import math
import re
import sqlite3
import uuid
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Annotated, List, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    RemoveMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

# Cargar API Key gratuita de Google AI Studio
load_dotenv()
api_key = os.environ["GEMINI_API_KEY"].strip().strip('"').strip("'")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=api_key,
    temperature=0.1,
)

# Asegurar entorno de datos de negocio (ops_company.db y SKILL.md)
os.makedirs("skills/ops_procurement", exist_ok=True)

skill_path = Path("skills/ops_procurement/SKILL.md")
if not skill_path.exists():
    skill_path.write_text(
        """---
name: ops_procurement
description: Procedimiento estándar para auditar desviaciones de ventas regionales y generar propuestas de compra de equipamiento.
---

# Reglas de Negocio y Aprobación
1. **Criterio de desviación:** Una región se considera en alerta crítica si sus ingresos están más de un 10% por debajo del objetivo trimestral.
2. **Límites presupuestarios de equipamiento:**
   - Hasta 3.000 EUR: Aprobación estándar de Responsable de Operaciones.
   - Más de 3.000 EUR: Requiere escalado obligatorio a Dirección Financiera (CFO) en el borrador.
3. **Stock:** Nunca proponer una compra que supere el stock disponible en catálogo.

# Plantilla Obligatoria del Informe Final
## 1. Diagnóstico Regional
- Región afectada, ingresos vs. objetivo y desviación porcentual.
## 2. Propuesta de Equipamiento y Cumplimiento
- Detalle de SKU, cantidad, coste total y nivel de aprobación requerido.
## 3. Borrador de Orden Generado
- ID del borrador y estado de validación.
""",
        encoding="utf-8",
    )

conn = sqlite3.connect("ops_company.db")
cur = conn.cursor()
cur.execute(
    """CREATE TABLE IF NOT EXISTS regional_sales (
        region TEXT, quarter TEXT, revenue_eur REAL, target_eur REAL, headcount INTEGER
    )"""
)
cur.execute("DELETE FROM regional_sales")
cur.executemany(
    "INSERT INTO regional_sales VALUES (?, ?, ?, ?, ?)",
    [
        ("Norte", "Q3", 420000, 400000, 18),
        ("Sur", "Q3", 265000, 350000, 14),
        ("Centro", "Q3", 510000, 500000, 22),
        ("Levante", "Q3", 310000, 320000, 15),
    ],
)
cur.execute(
    """CREATE TABLE IF NOT EXISTS hardware_catalog (
        sku TEXT PRIMARY KEY, item_name TEXT, unit_price_eur REAL, stock INTEGER
    )"""
)
cur.execute("DELETE FROM hardware_catalog")
cur.executemany(
    "INSERT INTO hardware_catalog VALUES (?, ?, ?, ?)",
    [
        ("LAP-PRO-16", "Portátil Pro 16GB RAM", 1250.0, 10),
        ("LAP-DEV-32", "Portátil Workstation 32GB RAM", 2100.0, 4),
        ("MON-27-4K", "Monitor 27 pulgadas 4K", 380.0, 15),
    ],
)
cur.execute(
    """CREATE TABLE IF NOT EXISTS purchase_drafts (
        draft_id TEXT PRIMARY KEY,
        thread_id TEXT,
        region TEXT,
        sku TEXT,
        quantity INTEGER,
        total_eur REAL,
        approval_level TEXT,
        status TEXT,
        created_at TEXT
    )"""
)
conn.commit()
conn.close()
print("Entorno LangGraph + Gemini + SQLite inicializado correctamente.")

## Hito 1: Memoria a Largo Plazo y Context Engineering (Celdas 2 y 3)

Antes de construir el grafo, dotaremos al sistema de dos capacidades esenciales:
- **Almacén de Memoria a Largo Plazo (`LongTermMemoryStore`):** Persiste en `long_term_memory.db` recuerdos **semánticos** (reglas/preferencias de negocio) y **episódicos** (órdenes aprobadas en el pasado) para que estén disponibles en cualquier sesión futura (`thread_id`).
- **Motor de Context Engineering (`compress_medium_term_context`):** Evita que el historial de mensajes crezca sin control resumiendo los turnos antiguos en un `rolling_summary` y eliminándolos de la ventana activa con `RemoveMessage`.

In [ ]:
# Memoria a Largo Plazo (Semántica + Episódica en SQLite)
class ExtractedMemory(BaseModel):
    memory_type: Literal["semantic", "episodic"] = Field(
        description="'semantic' para reglas, hechos o preferencias permanentes; 'episodic' para decisiones pasadas, compras o incidentes ocurridos."
    )
    content: str = Field(description="Descripción concisa y autocontenida del recuerdo.")
    importance: int = Field(ge=1, le=5, description="Importancia de 1 a 5.")

class ExtractedMemoryList(BaseModel):
    memories: List[ExtractedMemory]

class LongTermMemoryStore:
    """Almacena y recupera recuerdos semánticos y episódicos entre sesiones distintas."""

    def __init__(self, db_path: str = "long_term_memory.db"):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                """CREATE TABLE IF NOT EXISTS memories (
                    id TEXT PRIMARY KEY,
                    user_id TEXT,
                    memory_type TEXT,
                    content TEXT,
                    importance INTEGER,
                    created_at TEXT
                )"""
            )

    @staticmethod
    def _tokenize(text: str) -> Counter:
        tokens = re.findall(r"\b[a-záéíóúñ0-9\-]{3,}\b", text.lower())
        return Counter(tokens)

    def _cosine_similarity(self, text_a: str, text_b: str) -> float:
        vec_a, vec_b = self._tokenize(text_a), self._tokenize(text_b)
        if not vec_a or not vec_b:
            return 0.0
        common = set(vec_a.keys()) & set(vec_b.keys())
        dot = sum(vec_a[k] * vec_b[k] for k in common)
        norm_a = math.sqrt(sum(v * v for v in vec_a.values()))
        norm_b = math.sqrt(sum(v * v for v in vec_b.values()))
        return dot / (norm_a * norm_b)

    def save_memory(self, user_id: str, memory_type: str, content: str, importance: int = 3):
        with sqlite3.connect(self.db_path) as conn:
            exists = conn.execute(
                "SELECT 1 FROM memories WHERE user_id = ? AND content = ?",
                (user_id, content),
            ).fetchone()
            if not exists:
                conn.execute(
                    "INSERT INTO memories VALUES (?, ?, ?, ?, ?, ?)",
                    (
                        f"MEM-{uuid.uuid4().hex[:6].upper()}",
                        user_id,
                        memory_type,
                        content,
                        importance,
                        datetime.now().isoformat(),
                    ),
                )

    def retrieve(self, user_id: str, query: str, top_k: int = 4) -> List[dict]:
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                "SELECT * FROM memories WHERE user_id = ?", (user_id,)
            ).fetchall()

        # TODO 1: Para cada fila en `rows`, calcula su relevancia combinando:
        #   sim = self._cosine_similarity(query, row["content"])
        #   score = sim + (row["importance"] * 0.05)
        # Ordena los resultados de mayor a menor `score` y devuelve los `top_k` diccionarios.
        scored = []
        # Escribe aquí tu código para TODO 1...
        pass

    def consolidate_from_interaction(self, user_id: str, interaction_summary: str):
        """Usa el LLM para destilar nuevos recuerdos semánticos y episódicos tras cerrar una tarea."""
        # TODO 2: Crea un extractor estructurado con `llm.with_structured_output(ExtractedMemoryList)`,
        # invócalo pidiéndole extraer recuerdos 'semantic' o 'episodic' de `interaction_summary`,
        # y guarda cada recuerdo extraído llamando a `self.save_memory(...)`.
        # Escribe aquí tu código para TODO 2...
        pass

# Inicializamos el almacén y cargamos recuerdos históricos previos de la empresa
ltm_store = LongTermMemoryStore()
ltm_store.save_memory(
    "director_ops",
    "semantic",
    "Preferencia de Dirección: Para refuerzos comerciales en regiones críticas usar siempre el modelo LAP-PRO-16, reservando LAP-DEV-32 exclusivamente para perfiles de ingeniería.",
    importance=5,
)
ltm_store.save_memory(
    "director_ops",
    "episodic",
    "Histórico Q2: Se aprobó la compra de 2 monitores MON-27-4K para Levante (DRAFT-Q2001); el proveedor tardó 7 días hábiles en entregar.",
    importance=3,
)
print("Almacén de Memoria a Largo Plazo inicializado.")

In [ ]:
# Context Engineering (Rolling Summary + Window Trimming)
MAX_SHORT_TERM_MESSAGES = 4  # Mantenemos solo los últimos 4 mensajes crudos en ventana activa

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def compress_medium_term_context(
    messages: List[BaseMessage], current_summary: str
) -> tuple[str, List[RemoveMessage]]:
    """
    Evita que la ventana de contexto explote en conversaciones largas:
    1. Si `len(messages) <= MAX_SHORT_TERM_MESSAGES`, devuelve `(current_summary, [])`.
    2. Si hay más mensajes, toma `messages[:-MAX_SHORT_TERM_MESSAGES]`, pide al `llm` que actualice
       `current_summary` sin perder cifras, SKUs ni decisiones, y devuelve `(new_summary, removals)`
       donde `removals` es una lista de `RemoveMessage(id=m.id)` para los mensajes compactados.
    """
    # TODO 3: Implementa la lógica de compresión y poda de ventana activa.
    # PISTA: Asegúrate de comprobar `if m.id is not None` al crear cada `RemoveMessage(id=m.id)`.
    pass

print("Motor de Context Engineering listo.")

## Hitos 2 y 3: Grafo Cíclico con Autocorrección, Persistencia e HITL (Celda 4)

Completa los `TODO 4` a `TODO 7` en la siguiente celda para ensamblar el `StateGraph`:
- **`auditor_node` (`TODO 4`):** Evalúa el borrador generado por el `Drafter` frente a `SKILL.md`, el stock real y la memoria semántica.
- **`route_after_audit` (`TODO 5`):** Implementa el bucle cíclico `drafter <-> auditor` con un máximo de `3` intentos.
- **`human_approval_node` (`TODO 6`):** Detiene el flujo con `interrupt()`, guarda la orden aprobada en SQLite y consolida memoria episódica.
- **Compilación (`TODO 7`):** Conecta las aristas (incluyendo `add_conditional_edges`) y compila con `SqliteSaver`.

In [ ]:
# Grafo Cíclico (StateGraph) + Persistencia + HITL

# Definición del Estado Persistente del Grafo
class CopilotState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_id: str
    rolling_summary: str
    retrieved_memories: str
    business_metrics: str
    skill_rules: str
    draft_proposal: dict
    final_report: str
    audit_passed: bool
    audit_feedback: str
    retry_count: int
    order_status: str

# Esquemas Pydantic para salidas estructuradas
class DraftOutput(BaseModel):
    region: str
    sku: str
    quantity: int
    unit_price_eur: float
    total_eur: float
    approval_level: str
    markdown_report: str

class AuditVerdict(BaseModel):
    passed: bool = Field(
        description="True solo si cumple TODAS las reglas de SKILL.md, stock y memoria semántica."
    )
    issues: List[str] = Field(
        description="Lista de incumplimientos detectados (vacía si passed=True)."
    )
    fix_instructions: str = Field(
        description="Instrucciones concretas para corregir el borrador en el siguiente ciclo."
    )

# Nodos del Grafo
def context_and_memory_node(state: CopilotState) -> dict:
    """Paso 1: Comprime contexto a medio plazo y recupera memoria semántica/episódica a largo plazo."""
    messages = state.get("messages", [])
    current_summary = state.get("rolling_summary", "")
    user_id = state.get("user_id", "director_ops")

    new_summary, removals = compress_medium_term_context(messages, current_summary)

    latest_query = messages[-1].content if messages else ""
    memories = ltm_store.retrieve(user_id=user_id, query=latest_query, top_k=3) or []
    formatted_memories = "\n".join(
        f"- [{m['memory_type'].upper()}] {m['content']}" for m in memories
    ) or "Sin recuerdos previos relevantes."

    print(f"[Context & Memory] Mensajes activos: {len(messages) - len(removals)} | Recuerdos recuperados: {len(memories)}")
    return {
        "messages": removals,
        "rolling_summary": new_summary,
        "retrieved_memories": formatted_memories,
        "retry_count": 0,
        "audit_feedback": "",
    }

def data_analyst_node(state: CopilotState) -> dict:
    """Paso 2: Consulta SQLite de ventas y catálogo de hardware."""
    with sqlite3.connect("ops_company.db") as conn:
        conn.row_factory = sqlite3.Row
        sales = [dict(r) for r in conn.execute("SELECT * FROM regional_sales WHERE quarter = 'Q3'").fetchall()]
        catalog = [dict(r) for r in conn.execute("SELECT * FROM hardware_catalog").fetchall()]

    for s in sales:
        s["deviation_pct"] = round(((s["revenue_eur"] - s["target_eur"]) / s["target_eur"]) * 100, 2)

    skill_text = Path("skills/ops_procurement/SKILL.md").read_text(encoding="utf-8")
    metrics_json = json.dumps({"sales_q3": sales, "hardware_catalog": catalog}, ensure_ascii=False, indent=2)

    print("[Data Analyst] Métricas Q3 y catálogo cargados desde ops_company.db.")
    return {
        "business_metrics": metrics_json,
        "skill_rules": skill_text,
    }

def drafter_node(state: CopilotState) -> dict:
    """Paso 3 (Nodo cíclico): Genera o corrige la propuesta usando SKILL.md, memoria y feedback del Auditor."""
    attempt = state.get("retry_count", 0) + 1
    feedback = state.get("audit_feedback", "")
    latest_request = state["messages"][-1].content

    print(f"[Drafter] Generando borrador e informe (Iteración #{attempt})...")

    structured_drafter = llm.with_structured_output(DraftOutput)
    prompt = (
        "Eres el especialista de Operaciones y Compras. Genera una propuesta de compra y un informe ejecutivo "
        "cumpliendo estrictamente la Skill corporativa y los recuerdos a largo plazo.\n\n"
        f"PETICIÓN DEL USUARIO:\n{latest_request}\n\n"
        f"RESUMEN CONVERSACIÓN PREVIA (Medio Plazo):\n{state.get('rolling_summary', 'N/A')}\n\n"
        f"MEMORIA A LARGO PLAZO (Semántica y Episódica):\n{state['retrieved_memories']}\n\n"
        f"DATOS REALES DE NEGOCIO (SQLite):\n{state['business_metrics']}\n\n"
        f"SKILL CORPORATIVA OBLIGATORIA:\n{state['skill_rules']}\n\n"
    )
    if feedback:
        prompt += (
            "ATENCIÓN — TU BORRADOR ANTERIOR FUE RECHAZADO POR EL AUDITOR.\n"
            f"CORRIGE OBLIGATORIAMENTE ESTOS PUNTOS:\n{feedback}\n"
        )

    draft: DraftOutput = structured_drafter.invoke(prompt)
    draft_dict = draft.model_dump()
    draft_dict["draft_id"] = f"DRAFT-{uuid.uuid4().hex[:6].upper()}"

    return {
        "draft_proposal": draft_dict,
        "final_report": draft.markdown_report,
        "retry_count": attempt,
    }

def auditor_node(state: CopilotState) -> dict:
    """Paso 4 (Nodo Critic): Audita el borrador contra SKILL.md, stock y memoria semántica."""
    # TODO 4: Usa `llm.with_structured_output(AuditVerdict)` para evaluar `state['draft_proposal']`
    # frente a `state['business_metrics']`, `state['retrieved_memories']` y `state['skill_rules']`.
    # Si `verdict.passed` es True, devuelve `{"audit_passed": True, "audit_feedback": ""}`.
    # Si es False, devuelve `{"audit_passed": False, "audit_feedback": f"Problemas: {verdict.issues}. Instrucciones: {verdict.fix_instructions}"}`.
    pass

def route_after_audit(state: CopilotState) -> Literal["human_approval", "drafter"]:
    """Arista condicional cíclica: reintenta hasta 3 veces si el Auditor rechaza el borrador."""
    # TODO 5: Implementa la decisión de enrutamiento:
    # - Si `state["audit_passed"]` es True -> "human_approval"
    # - Si `state["retry_count"] < 3` -> "drafter"
    # - En caso contrario (agotados los 3 intentos) -> "human_approval"
    pass

def human_approval_node(state: CopilotState) -> dict:
    """Paso 5: Pausa durable (HITL) mediante interrupt(). Guarda la orden en SQLite si se aprueba."""
    draft = state["draft_proposal"]

    # TODO 6:
    # 1. Llama a `human_decision = interrupt({...})` pasando un diccionario con los datos clave del borrador
    #    (draft_id, region, sku, quantity, total_eur, approval_level, audit_passed).
    # 2. Si `str(human_decision).strip().lower() == "s"`:
    #    - Inserta el registro en la tabla `purchase_drafts` de `ops_company.db` con estado "APPROVED_AND_PERSISTED".
    #    - Llama a `ltm_store.consolidate_from_interaction(...)` para guardar esta decisión en memoria episódica.
    #    - Devuelve `{"order_status": "APPROVED_AND_PERSISTED", "messages": [AIMessage(content=...)]}`.
    # 3. Si no se aprueba, devuelve `{"order_status": "REJECTED_BY_HUMAN", "messages": [AIMessage(content=...)]}`.
    pass

# Construcción y compilación del Grafo Cíclico con Checkpointer Persistente
builder = StateGraph(CopilotState)
builder.add_node("context_and_memory", context_and_memory_node)
builder.add_node("data_analyst", data_analyst_node)
builder.add_node("drafter", drafter_node)
builder.add_node("auditor", auditor_node)
builder.add_node("human_approval", human_approval_node)

# TODO 7: Conecta las aristas del grafo:
# START -> context_and_memory -> data_analyst -> drafter -> auditor
# Desde "auditor" usa `builder.add_conditional_edges("auditor", route_after_audit, {"drafter": "drafter", "human_approval": "human_approval"})`
# human_approval -> END
# Finalmente crea `checkpoint_conn = sqlite3.connect("langgraph_checkpoints.db", check_same_thread=False)`,
# instancia `checkpointer = SqliteSaver(checkpoint_conn)` y compila:
# `copilot_graph = builder.compile(checkpointer=checkpointer)`

print("Grafo cíclico compilado correctamente.")

## Validación Experimental (Celdas 5, 6 y 7)

Ejecuta las tres celdas siguientes en orden para comprobar:
1. **Celda 5:** Cómo el grafo detecta el incumplimiento de memoria semántica (`LAP-DEV-32` prohibido para comerciales), entra en el bucle de autocorrección hacia `LAP-PRO-16` y se detiene en `interrupt()`.
2. **Celda 6:** Prueba de fuego de persistencia: **reinicia el Kernel de VS Code**, vuelve a ejecutar las Celdas 1–4 y lanza directamente la Celda 6 con `Command(resume="s")`.
3. **Celda 7:** Verificación de memoria episódica en un hilo nuevo (`ops-thread-002`) y simulación de compresión de contexto a medio plazo.

In [ ]:
# Demo 1 — Ciclo de Autocorrección + Pausa Persistente (interrupt)
config_sesion_1 = {"configurable": {"thread_id": "ops-thread-001"}}

result = copilot_graph.invoke(
    {
        "user_id": "director_ops",
        "messages": [
            HumanMessage(
                content="Analiza Q3, detecta la región en alerta crítica y prepara compra de 3 portátiles workstation LAP-DEV-32 para reforzar a sus comerciales."
            )
        ],
    },
    config=config_sesion_1,
)

print("\n--- ESTADO PAUSADO EN INTERRUPT (HITL) ---")
print(result.get("__interrupt__"))

In [ ]:
# Demo 2 — Reanudación tras Pausa (funciona incluso tras reiniciar Kernel)
config_sesion_1 = {"configurable": {"thread_id": "ops-thread-001"}}

# Inspeccionamos el historial de checkpoints guardados en SQLite (Time-Travel)
history = list(copilot_graph.get_state_history(config_sesion_1))
print(f"Checkpoints guardados en SQLite para este hilo: {len(history)}")

# Reanudamos la ejecución inyectando la aprobación humana 's'
final_state = copilot_graph.invoke(Command(resume="s"), config=config_sesion_1)

print("\n=== INFORME FINAL TRAS APROBACIÓN ===")
print(final_state["messages"][-1].content)

In [ ]:
# Demo 3 — Memoria entre Sesiones Distintas + Context Engineering
# Comprobamos si en una sesión nueva se recuerda la compra aprobada en 'ops-thread-001'
memories_for_new_session = ltm_store.retrieve(
    user_id="director_ops",
    query="¿Qué órdenes de compra aprobamos recientemente para la región Sur?",
    top_k=3,
) or []

print("=== MEMORIA A LARGO PLAZO RECUPERADA EN UNA NUEVA SESIÓN ===")
for m in memories_for_new_session:
    print(f"- [{m['memory_type'].upper()}] {m['content']}")

# Simulamos una conversación larga de 7 mensajes para observar la compresión de medio plazo
long_history = [
    HumanMessage(id=f"msg-{i}", content=f"Consulta operativa #{i}: revisando métricas parciales de la región Sur.")
    for i in range(1, 8)
]
summary, removals = compress_medium_term_context(long_history, "")
print("\n=== CONTEXT ENGINEERING (MEDIO PLAZO) ===")
print(f"Mensajes purgados de la ventana activa: {len(removals)} de {len(long_history)}")
print(f"Rolling Summary generado ({estimate_tokens(summary)} tokens aprox.):\n{summary}")

---
## Retos de Ampliación (Opcional para profundizar)

Si has completado todos los hitos anteriores, elige uno de estos tres retos de ingeniería en producción:

1. **Reto 1 — *Time-Travel* real (rebobinar y bifurcar una decisión):**
   Inspecciona `history = list(copilot_graph.get_state_history(config_sesion_1))`, elige un checkpoint previo a `drafter`, modifica el estado con `copilot_graph.update_state(checkpoint.config, {"audit_feedback": "Presupuesto máximo autorizado hoy: 2.500 EUR. Ajusta a 2 unidades."})` y relanza `copilot_graph.invoke(None, checkpoint.config)`.
2. **Reto 2 — Decaimiento temporal (*Memory Decay*):**
   Modifica `LongTermMemoryStore.retrieve()` para penalizar ligeramente los recuerdos más antiguos usando `created_at`.
3. **Reto 3 — Prueba de límite de reintentos (*Graceful Fallback*):**
   Lanza un hilo nuevo (`ops-thread-003`) pidiendo **50 portátiles `LAP-PRO-16` sin permitir reducir la cantidad** y observa cómo el grafo agota los 3 ciclos y escala al humano con `audit_passed=False`.

In [ ]:
# Espacio libre para experimentar con los Retos de Ampliación:
